# Practice Session PS06: Interactions-based recommendation

<font size="+2" color="#2680EB">Additional results: surprise library</font> 

Done at the end of the notebook

Author: <font color="#2680EB">Anna Quílez Rodríguez</font>

E-mail: <font color="#2680EB">anna.quilez01@estudiant.upf.edu</font>

AI used: <font color="#2680EB">ChatGPT. I used artificial intelligence to help me understand the functions I wasn’t familiar with and to write some of the code.</font>

Date: <font color="#2680EB">4/11/2026</font>

# 1. The Movies dataset

# 1.1. Load the input files

In [82]:
# LEAVE THIS CODE AS-IS
# But feel free to add imports in an extra cell if needed

import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
from math import *
import random
from scipy.sparse.linalg import svds
from sklearn.metrics.pairwise import linear_kernel

In [ ]:
# LEAVE THIS CODE AS-IS

FILENAME_MOVIES = "ml32m-ratings-2000s.csv.gz"
FILENAME_RATINGS = "ml32m-ratings-2000s.csv.gz"

In [84]:
# LEAVE AS-IS

# Load movies
movies = pd.read_csv(FILENAME_MOVIES, 
                    compression='gzip',
                    sep=',', 
                    engine='python', 
                    encoding='utf-8',
                    names=['movie_id', 'title', 'genres'])

# Remove header row from this file
movies.drop(index=0, inplace=True)

# Fix encoding issues in title
# The instructor used AI to fix this encoding issue (Gemini 3.6 Flash)
movies['title'] = movies['title'].str.encode('latin1').str.decode('utf-8')

# Make sure the movie id is numeric
movies["movie_id"] = pd.to_numeric(movies["movie_id"])
print(f"Loaded {movies.shape[0]} movies")
display(movies.sample(10))

Loaded 51444 movies


,movie_id,title,genres
24358,178255,No Subtitles Necessary: Laszlo & Vilmos (2009),Documentary
49250,282919,Additional Lesson (2022),Drama
11759,128912,Mike Birbiglia: My Girlfriend's Boyfriend (2013),Comedy
25418,181687,The Wednesday (2016),Children|Crime|Drama
17142,152358,Jaani Dushman: Ek Anokhi Kahani (2002),(no genres listed)
11202,125135,Unstable Fables: Tortoise vs. Hare (2008),Animation|Children|Drama
44950,264095,Surviving the Outback (2019),Documentary
17004,152009,Supervolcano (2005),Action|Drama|Thriller
20172,164600,Akira (2016),Action|Crime|Thriller
45455,266840,Echoes of the Past (2021),Drama


In [85]:
# LEAVE AS-IS

# Load ratings
ratings_raw = pd.read_csv(FILENAME_RATINGS, 
                    sep=',', 
                    compression='gzip',
                    encoding='utf-8',
                    engine='python',
                    names=['user_id', 'movie_id', 'rating'])
display(ratings_raw.head(5))

,user_id,movie_id,rating
0,4,223,4.0
1,4,1210,3.0
2,4,1272,4.0
3,4,1327,3.0
4,4,1513,2.0


In [86]:
# LEAVE AS-IS

# Print some stats on the ratings data
n_users = ratings_raw.user_id.unique().shape[0]
n_movies = ratings_raw.movie_id.unique().shape[0]
print(f"Users who have rated a movie : {n_users}")
print(f"Movies that have been rated  : {n_movies} of {movies.shape[0]}")

Users who have rated a movie : 21940
Movies that have been rated  : 10938 of 51444


# 1.2. Merge the data into a single dataframe <font color="#2680EB">[5%]</font>

In [87]:

# YOUR CODE HERE

# I join the ratings with the movie information using movie_id.
ratings = pd.merge(ratings_raw,movies[['movie_id', 'title', 'genres']],how='inner',on='movie_id')
#Join data. we use "inner" because we want to retain only rows in both DataFrames.
# There is no timestamp column in the ratings dataset.

ratings.head(5)

,user_id,movie_id,rating,title,genres
0,33,3285,4.5,"Beach, The (2000)",Adventure|Drama
1,33,3535,4.5,American Psycho (2000),Crime|Horror|Mystery|Thriller
2,33,3578,4.5,Gladiator (2000),Action|Adventure|Drama
3,33,3617,1.5,Road Trip (2000),Comedy
4,33,3623,4.0,Mission: Impossible II (2000),Action|Adventure|Thriller


<font size="+1" color="#2680EB">**Copy your implementation of find_movies from the previous practice.** Note that in this dataset the id of a movie is named *movie_id*.</font>

In [88]:
def find_movies(keyword, movies):
    # YOUR CODE HERE
    # I go through each row of the movies DataFrame.
    for i, movie in movies.iterrows():

        # I check if the keyword is in the movie title.
        # lower() makes the search case-insensitive.
        if keyword.lower() in str(movie["title"]).lower():
            # I print the movie ID and its title.
            print(f"id: {movie['movie_id']}, title: {movie['title']}")
    
find_movies("Rouge", movies)

id: 4308, title: Moulin Rouge (2001)
id: 5537, title: Satin Rouge (2002)
id: 27776, title: Red Lights (Feux rouges) (2004)
id: 84114, title: S21: The Khmer Rouge Death Machine (S-21, la machine de mort Khmère rouge) (2003)
id: 109010, title: Courage of the Eagles, The (Les aiguilles rouges) (2006)
id: 129931, title: La Morte Rouge (2006)
id: 132078, title: Hiver rouge (2011)
id: 142513, title: Hunter × Hunter: Phantom Rouge (2013)
id: 175883, title: Les fourmis rouges (2007)
id: 180101, title: Benoît Brisefer : Les taxis rouges (2014)
id: 208529, title: Staline: Le tyran rouge (2007)


In [89]:
# LEAVE AS-IS

def get_title(movie_id, movies):
    return movies[movies['movie_id'] == movie_id].title.iloc[0]

def get_genres(movie_id, movies):
    return movies[movies['movie_id'] == movie_id].genres.iloc[0]

def get_movie_id_by_title(title, movies):
    match = movies.loc[movies['title'] == title, 'movie_id']
    return match.iloc[0] if not match.empty else None

# Should find the movie "Idiocracy (2006)"
print(f"Title of movie 47997  : {get_title(47997, movies)}")
print(f"Genres of movie 47997 : {get_genres(47997, movies)}")
print(f"ID of movie 'Idiocracy (2006)': {get_movie_id_by_title("Idiocracy (2006)", movies)}")

Title of movie 47997  : Idiocracy (2006)
Genres of movie 47997 : Adventure|Comedy|Sci-Fi|Thriller
ID of movie 'Idiocracy (2006)': 47997


# 2. Item-based Collaborative Filtering

## 2.1. Data pre-processing <font color="#2680EB">[15%]</font>

<font size="+1" color="#2680EB">**Generate ratings_summary.**</font>

In [ ]:
ratings_grouped = ratings.groupby(['movie_id', 'title', 'genres'])

# YOUR CODE HERE
ratings_summary = ratings_grouped['rating'].agg(ratings_mean='mean',ratings_median='median',ratings_count='count').reset_index()
# I use agg() to apply different aggregation functions to the rating column.
# I convert the result into a regular DataFrame.


display(ratings_summary.sample(10))

,movie_id,title,genres,ratings_mean,ratings_median,ratings_count
444,4869,Burnt Money (Plata Quemada) (2000),Action|Crime|Drama|Romance|Thriller,2.333333,2.5,3
963,6291,Lilya 4-Ever (Lilja 4-ever) (2002),Crime|Drama,3.766082,4.0,171
2081,33649,Saving Face (2004),Comedy|Drama|Romance,3.885714,4.0,35
2625,47394,"Bon Cop, Bad Cop (2006)",Action|Comedy|Crime|Thriller,3.318182,3.5,22
2578,46559,"Road to Guantanamo, The (2006)",Drama|War,3.466667,4.0,15
1394,7769,Legend of the Village Warriors (Bangrajan) (2000),Action|Drama|War,3.500000,4.0,4
692,5513,Martin Lawrence Live: Runteldat (2002),Comedy|Documentary,2.500000,2.5,14
129,3892,Anatomy (Anatomie) (2000),Horror,3.156250,3.5,48
2249,38388,Goal! The Dream Begins (Goal!) (2005),Drama,3.310000,3.5,50
1931,31613,Easy (2003),Comedy|Drama|Romance,2.625000,3.0,4


<font size="+1" color="#2680EB">**Print the best and worst movies.** Remember to consider only movies having at least 1000 ratings.</font>

In [91]:
def get_best_movies(min_ratings, max_movies=10):
    
    # YOUR CODE HERE
    filtered_movies = ratings_summary[ratings_summary['ratings_count'] >= min_ratings] # I keep only movies with at least the specified number of ratings
    best_movies = filtered_movies.sort_values(by=['ratings_median', 'ratings_mean'],ascending=[False, False]).head(max_movies)
    # I sort movies by median and then by mean, in descending order
    #ascending=False means that the highest values appear first
    # I select the first max_movies movies suing head()
    
    return best_movies

def get_worst_movies(min_ratings, max_movies=10):
    # YOUR CODE HERE
    filtered_movies = ratings_summary[ratings_summary['ratings_count'] >= min_ratings]  # I keep only movies with at least the specified number of ratings
    worst_movies = filtered_movies.sort_values(by=['ratings_median', 'ratings_mean'],ascending=[True, True]).head(max_movies)
    # I sort movies by median and mean, in ascending order, ascending=True 
    # I select the first max_movies movies suing head()
    return worst_movies

min_ratings = 1000
print(f"Best movies by rating (at least {min_ratings} ratings)")
display(get_best_movies(min_ratings))
print()
print(f"Worst movies by rating (at least {min_ratings} ratings)")
display(get_worst_movies(min_ratings))

Best movies by rating (at least 1000 ratings)


,movie_id,title,genres,ratings_mean,ratings_median,ratings_count
736,5618,Spirited Away (Sen to Chihiro no kamikakushi) ...,Adventure|Animation|Fantasy,4.224841,4.5,3607
2469,44555,"Lives of Others, The (Das leben der Anderen) (...",Drama|Romance|Thriller,4.211582,4.5,1321
882,6016,City of God (Cidade de Deus) (2002),Action|Adventure|Crime|Drama|Thriller,4.187149,4.5,2848
1255,7153,"Lord of the Rings: The Return of the King, The...",Action|Adventure|Drama|Fantasy,4.091107,4.5,7365
498,4993,"Lord of the Rings: The Fellowship of the Ring,...",Adventure|Fantasy,4.089918,4.5,7935
259,4226,Memento (2000),Mystery|Thriller,4.158425,4.0,5804
2692,48516,"Departed, The (2006)",Crime|Drama|Thriller,4.138387,4.0,3819
2718,48780,"Prestige, The (2006)",Drama|Mystery|Sci-Fi|Thriller,4.104571,4.0,3610
1934,31658,Howl's Moving Castle (Hauru no ugoku shiro) (2...,Adventure|Animation|Fantasy|Romance,4.091309,4.0,1703
487,4973,"Amelie (Fabuleux destin d'Amélie Poulain, Le) ...",Comedy|Romance,4.069024,4.0,4723



Worst movies by rating (at least 1000 ratings)


,movie_id,title,genres,ratings_mean,ratings_median,ratings_count
89,3785,Scary Movie (2000),Comedy|Horror,2.702917,3.0,1577
168,3977,Charlie's Angels (2000),Action|Comedy,2.724966,3.0,2227
362,4643,Planet of the Apes (2001),Action|Adventure|Drama|Sci-Fi,2.754444,3.0,1350
325,4367,Lara Croft: Tomb Raider (2001),Action|Adventure,2.760097,3.0,1238
357,4638,Jurassic Park III (2001),Action|Adventure|Sci-Fi|Thriller,2.778367,3.0,1225
310,4310,Pearl Harbor (2001),Action|Drama|Romance|War,2.869378,3.0,1512
177,3988,How the Grinch Stole Christmas (a.k.a. The Gri...,Children|Comedy|Fantasy,2.872857,3.0,1050
371,4718,American Pie 2 (2001),Comedy,2.876039,3.0,1444
327,4369,"Fast and the Furious, The (2001)",Action|Crime|Thriller,2.962895,3.0,1361
674,5481,Austin Powers in Goldmember (2002),Comedy,2.983943,3.0,1557


In [92]:
# LEAVE AS-IS

min_ratings = 3
print(f"Best movies by rating (at least {min_ratings} ratings)")
display(get_best_movies(min_ratings))
print()
print(f"Worst movies by rating (at least {min_ratings} ratings)")
display(get_worst_movies(min_ratings))

Best movies by rating (at least 3 ratings)


,movie_id,title,genres,ratings_mean,ratings_median,ratings_count
248,4165,"Me You Them (Eu, Tu, Eles) (2000)",Comedy|Drama|Romance,4.250000,4.75,6
2143,34314,Funny Ha Ha (2002),Comedy|Drama,3.875000,4.75,4
1711,27550,Hell House (2001),Documentary,4.500000,4.50,3
1958,31900,Travellers and Magicians (2003),Adventure|Drama,4.300000,4.50,5
1393,7767,"Best of Youth, The (La meglio gioventù) (2003)",Drama,4.250000,4.50,54
1731,27627,Oasis (2002),Drama|Romance,4.250000,4.50,6
1739,27646,Soldier's Girl (2003),Drama,4.250000,4.50,6
1668,27423,"O Auto da Compadecida (Dog's Will, A) (2000)",Adventure|Comedy,4.242857,4.50,35
736,5618,Spirited Away (Sen to Chihiro no kamikakushi) ...,Adventure|Animation|Fantasy,4.224841,4.50,3607
1682,27469,Millennium Mambo (2001),Drama|Romance,4.214286,4.50,7



Worst movies by rating (at least 3 ratings)


,movie_id,title,genres,ratings_mean,ratings_median,ratings_count
611,5326,"Frank McKlusky, C.I. (2002)",Comedy,0.500000,0.5,3
1299,7282,"Hip Hop Witch, Da (2000)",Comedy|Horror|Thriller,0.500000,0.5,5
2562,45994,National Lampoon's Cattle Call (Cattle Call) (...,Comedy,0.500000,0.5,3
1529,8859,SuperBabies: Baby Geniuses 2 (2004),Comedy,0.806452,0.5,31
1021,6483,From Justin to Kelly (2003),Musical|Romance,0.850746,0.5,67
419,4781,Megiddo: The Omega Code 2 (2001),Action|Adventure|Fantasy|Sci-Fi|Thriller,1.300000,0.5,5
1356,7390,Prey for Rock & Roll (2003),Drama|Musical,1.500000,0.5,3
1005,6371,Pokémon Heroes (2003),Animation|Children,1.544643,0.5,56
1763,27719,Levottomat 3 (2004),Romance,1.000000,1.0,3
1944,31698,Son of the Mask (2005),Adventure|Children|Comedy|Fantasy,1.091837,1.0,98


<font size="+1" color="#2680EB">**Compare the lists with min_ratings=1000 against the lists with min_ratings=3.** Answer these questions.</font>

**Answer:**

What would you say about the lists above? 

* We can see that depenidng on the minimum number of ratings the lists differs. With at least 1000 ratings, the best movies include popular films such as Spirited Away, The Lord of the Rings and City of God. These movies have high ratings supported by a large number of users. However, when the minimum is reduced to just 3 ratings, some less-known movies appear at the top, even though they have only received 3 or 4 ratings. The same happens with the worst movies, where some films have very low ratings but very few users have rated them.

Why do you think this happens?

* I think this happens because the number of ratings affects how reliable the results are. When a movie has only 3 ratings, a few very high or very low scores can strongly affect its mean and median. This can make a movie appear much better or worse than it really is. For that reason, when we use at least 1000 ratings their results are generally more reliable because they are based on many more opinions.




## 2.2. Create the utility matrix <font color="#2680EB">[10%]</font>

<font size="+1" color="#2680EB">**Create the utility matrix.** Use the "pivot_table" function. This may take a few seconds, perhaps up to a minute, to compute.</font>

In [ ]:

# YOUR CODE HERE
#I create the utility matrix using pivot_table(), so each row represents a user, each column a movie and each cell contains the user's rating for that movie
user_movie = ratings.pivot_table(index='user_id',columns='movie_id',values='rating')
user_movie.head(10) 

movie_id,2769,3177,3190,3225,3228,3239,3273,3275,3276,3279,...,51187,51194,51255,51312,51314,51317,51402,51412,51418,51433
user_id,,,,,,,,,,,,,,,,,,,,,
33,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
63,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
94,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,4.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
95,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
131,NaN,NaN,NaN,0.5,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
132,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
140,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
152,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
162,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<font size="+1" color="#2680EB">**Notice that most cells are null.** Answer these questions.</font>

**Answer:**

Why do you think the `user_movie` matrix has so many null values?

* The utility matrix is sparse because there are a huge number of movies, but each user interacts with only a small fraction of them. Therefore, most entries in the utility matrix are unknown (null values).

How do we call this characteristic of user ratings in recommender systems?

* We call it sparsity. 


## 2.3. Explore some correlations in the user-movie matrix <font color="#2680EB">[20%]</font>

In [94]:
# LEAVE AS-IS

id_pivot = get_movie_id_by_title("Finding Nemo (2003)", movies)
id_m1 = get_movie_id_by_title("Animatrix, The (2003)", movies)
id_m2 = get_movie_id_by_title("Ratatouille (2007)", movies)

for movie_id in [id_pivot, id_m1, id_m2]:
    print(f"id: {movie_id:7d}, title: {get_title(movie_id, movies)}")

id:    6377, title: Finding Nemo (2003)
id:   27660, title: Animatrix, The (2003)
id:   50872, title: Ratatouille (2007)


<font size="+1" color="#2680EB">**Create the "ratings_anim_sample" dataframe.**</font>

In [107]:
# YOUR CODE HERE
#AI
# I get the ratings for each of the three movies.
# I use dropna() to keep only the existing ratings.
s1 = user_movie[id_pivot].dropna()
s2 = user_movie[id_m1].dropna()
s3 = user_movie[id_m2].dropna()

ratings_anim_sample = pd.concat([s1, s2, s3], axis=1) #join the three Series into one DataFrame, axis=1 to put each movie in a separate column.
ratings_anim_sample.columns = [get_title(id_pivot, movies), get_title(id_m1, movies), get_title(id_m2, movies)] # Assign the movie titles as column names
ratings_anim_sample = ratings_anim_sample.dropna() 

display(ratings_anim_sample)

,Finding Nemo (2003),"Animatrix, The (2003)",Ratatouille (2007)
user_id,,,
2244,5.0,3.5,5.0
2842,3.5,3.0,5.0
5902,3.0,3.0,4.0
6092,4.5,3.5,5.0
9158,4.5,3.5,4.0
...,...,...,...
199168,3.5,5.0,3.0
199478,3.0,0.5,3.0
199513,3.0,3.5,3.5


<font size="+1" color="#2680EB">**Answer the following questions.**</font>

**Answer:**

How many users have rated all the 3 movies?

* Due to there are 264 rows and each row represent an user, 264 users have rated all the 3 movies.

Do you think this is a usual or unusual number, in comparison with a random sample of 3 movies? Why?

* I think this is a relatively high number compared to a random sample of three movies. These movies may be quite popular, so many users have rated all of them. If we selected three movies randomly, it would probably be less common for users to have rated all three.

<font size="+1" color="#2680EB">**Compute pair-wise correlations between the ratings of the three movies.**</font>

In [ ]:

for pair in [(id_pivot, id_m1), (id_pivot, id_m2), (id_m1, id_m2)]:
    # YOUR CODE HERE
    # I calculate the Pearson correlation between the ratings of two movies.
    correlation = ratings_anim_sample[pair[0]].corr(ratings_anim_sample[pair[1]])

    print(f"Correlation between ratings of '{get_title(pair[0], movies)}' and '{get_title(pair[1], movies)}': {correlation:.4f}")
    #AI to discover the function get_title

Correlation between ratings of 'Finding Nemo (2003)' and 'Animatrix, The (2003)': 0.1709
Correlation between ratings of 'Finding Nemo (2003)' and 'Ratatouille (2007)': 0.5086
Correlation between ratings of 'Animatrix, The (2003)' and 'Ratatouille (2007)': 0.1517


<font size="+1" color="#2680EB">**Comment on the correlations found.** Why do you think one of them is higher than the others?</font>

**Answer:**

Why one of these correlations is higher than the others?

* The correlation between Finding Nemo and Ratatouille is higher (0.5086) because both movies are animated films and they have similar characteristics. Therefore, users who enjoy one of these movies may also enjoy the other. On the other hand, The Animatrix has a different style and genre, which could explain its lower correlations with the other two movies.

<font size="+1" color="#2680EB">**Create the "correlation_with_pivot" dataframe.** It should contain columns "movie_id" and "corr_with_pivot". </font>

In [97]:

# Obtain the ratings of the pivot
ratings_pivot = pd.DataFrame(user_movie[id_pivot].dropna()).rename(columns={id_pivot: "rating"})

correlation_with_pivot_arr = []

for id_movie in user_movie:
    # Obtain the ratings of the other movie
    # Append to correlation_with_pivot_arr an array [id_movie, correlation]
    #  where correlation is the computed correlation between ratings_pivot and the ratings of the other movie
    
    # YOUR CODE HERE
    ratings_movie = pd.DataFrame(user_movie[id_movie].dropna()).rename(columns={id_movie: "rating"}) # I get the ratings of the movie I want to compare
    correlation = ratings_pivot.corrwith(ratings_movie).iloc[0] # corrwith() compares the ratings from users who rated both movies
    correlation_with_pivot_arr.append([id_movie, correlation]) # I save the movie ID and its correlation

# Turn this into a dataframe
correlation_with_pivot = pd.DataFrame(correlation_with_pivot_arr, columns=["movie_id", "corr_with_pivot"])

# Drop null values
correlation_with_pivot = correlation_with_pivot.dropna(subset=['corr_with_pivot'])

# Display
display(correlation_with_pivot)

,movie_id,corr_with_pivot
0,2769,0.064023
1,3177,0.284792
2,3190,0.228716
3,3225,0.106047
5,3239,-0.347351
...,...,...
2871,51312,-0.458831
2872,51314,0.816497
2873,51317,1.000000
2875,51412,0.186027


<font size="+1" color="#2680EB">**Create the "correlation_with_pivot_info" dataframe.** Do it by performing an inner join between "correlation_with_pivot" and "ratings_summary".</font>

In [108]:
# YOUR CODE HERE
# I merge the correlations with the movie information using movie_id as the common column to join both DataFrames
correlation_with_pivot_info = pd.merge(correlation_with_pivot,ratings_summary, how='inner', on='movie_id')

# keep only movies with more than 1000 ratings
correlation_with_pivot_info = correlation_with_pivot_info[correlation_with_pivot_info['ratings_count'] > 1000]
print("Movies with high correlation")
display(correlation_with_pivot_info.sort_values('corr_with_pivot', ascending=False).head(5))

print("Movies with low correlation")
display(correlation_with_pivot_info.sort_values('corr_with_pivot').head(5))

Movies with high correlation


,movie_id,corr_with_pivot,title,genres,ratings_mean,ratings_median,ratings_count
916,6377,1.000000,Finding Nemo (2003),Adventure|Animation|Children|Comedy,3.826549,4.0,5036
427,4886,0.656471,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,3.837914,4.0,4985
1426,8961,0.556336,"Incredibles, The (2004)",Action|Adventure|Animation|Children|Comedy,3.824699,4.0,4575
2475,50872,0.549734,Ratatouille (2007),Animation|Children|Drama,3.825687,4.0,3095
518,5218,0.537767,Ice Age (2002),Adventure|Animation|Children|Comedy,3.537707,3.5,2652


Movies with low correlation


,movie_id,corr_with_pivot,title,genres,ratings_mean,ratings_median,ratings_count
1416,8949,0.041605,Sideways (2004),Comedy|Drama|Romance,3.675942,4.0,1168
403,4848,0.068701,Mulholland Drive (2001),Crime|Drama|Film-Noir|Mystery|Thriller,3.884509,4.0,1814
1653,30810,0.081858,"Life Aquatic with Steve Zissou, The (2004)",Adventure|Comedy|Fantasy,3.744598,4.0,1157
142,3949,0.091612,Requiem for a Dream (2000),Drama,3.975218,4.0,3208
1296,8376,0.094701,Napoleon Dynamite (2004),Comedy,3.398288,3.5,1519


<font size="+1" color="#2680EB">**Answer the questions below about these lists.**</font>

**Answer:**

How would you characterize the movies with high correlation?

* The movies with high correlation are mainly animated and family movies, such as Monsters, Inc., The Incredibles, Ratatouille and Ice Age. This suggests that users who enjoy Finding Nemo also tend to enjoy other animated movies. They may have similar characteristics and, for this reason, they may have a similar audience. 

One of the movies with low correlation seems to have a similar theme to the pivot movie, why do you think it has low correlation?

* Even if The Life Aquatic with Steve Zissou shares some adventure elements with Finding Nemo, they are very different types of movies. Finding Nemo is an animated family movie, whereas The Life Aquatic with Steve Zissou has a different style and is aimed at a different audience. For this reason, not all the users who enjoy Finding Nemo necessiarily enjoy the other movie, which could exaplain its lower correlation. 

# 3. Select two interesting users

In [99]:
# LEAVE AS-IS

# Gets the rating a user_id has given to a movie_id
def get_rating(user_movie, user_id, movie_id):
    # The instructor used AI to detect keys that could not be found and return a nan (Gemini 3.6 Flash)
    try:
        val = user_movie[movie_id][user_id]
        return None if pd.isna(val) else val
    except KeyError:
        return None

# Gets a list of movie_ids rated by a user
def get_rated_movie_ids(user_movie, user_id):
    return list(user_movie.loc[user_id].dropna().sort_values(ascending=False).index)

# Get rated movies as a Dataframe
def get_rated_movies(user_movie, movies, user_id):
    rated_movies = get_rated_movie_ids(user_movie, user_id)
    
    df_rated = pd.DataFrame([
        {
            'movie_id': movie_id,
            'rating': get_rating(user_movie, user_id, movie_id),
            'title': get_title(movie_id, movies),
            'genres': get_genres(movie_id, movies)
        } for movie_id in rated_movies
    ])
    return df_rated


## 3.1. Example selection <font color="#2680EB">[5%]</font>

In [100]:
# LEAVE AS-IS

# User who likes Harry Potter movies

hp_movie_1 = get_movie_id_by_title("Harry Potter and the Chamber of Secrets (2002)", movies)
hp_movie_2 = get_movie_id_by_title("Harry Potter and the Prisoner of Azkaban (2004)", movies)
hp_movie_3 = get_movie_id_by_title("Harry Potter and the Goblet of Fire (2005)", movies)

list_hp = user_movie[(user_movie[hp_movie_1] > 4.5)
                        & (user_movie[hp_movie_2] > 4.5)
                        & (user_movie[hp_movie_3] > 4.5)].index.values.tolist()
user_id_hp = random.choice(list_hp)

print("There are %d users who really like Harry Potter movies" % len(list_hp))
print("Selected user: user_id=%d" % user_id_hp)

# User who likes dark movies

dark_movie_1 = get_movie_id_by_title("Requiem for a Dream (2000)", movies)
dark_movie_2 = get_movie_id_by_title("American Psycho (2000)", movies)
dark_movie_3 = get_movie_id_by_title("Donnie Darko (2001)", movies)


list_dark = user_movie[(user_movie[dark_movie_1] > 4.5)
                        & (user_movie[dark_movie_2] > 4.5)
                        & (user_movie[dark_movie_3] > 4.5)
                        & (user_movie[hp_movie_1].isnull())].index.values.tolist()
user_id_dark = random.choice(list_dark)

print("There are %d users who like dark movies and have not rated Harry Potter" % len(list_dark))
print("Selected user: user_id=%d" % user_id_dark)


There are 252 users who really like Harry Potter movies
Selected user: user_id=27098
There are 21 users who like dark movies and have not rated Harry Potter
Selected user: user_id=112302


In [101]:
# LEAVE AS-IS

# We check that the selected Harry Potter fan is correct
assert get_rating(user_movie, user_id_hp, hp_movie_1) > 4.5
assert get_rating(user_movie, user_id_hp, hp_movie_2) > 4.5
assert get_rating(user_movie, user_id_hp, hp_movie_3) > 4.5

# We check that the selected dark movies fan is correct
assert get_rating(user_movie, user_id_dark, dark_movie_1) > 4.5
assert get_rating(user_movie, user_id_dark, dark_movie_2) > 4.5
assert get_rating(user_movie, user_id_dark, dark_movie_3) > 4.5
assert pd.isna(get_rating(user_movie, user_id_dark, hp_movie_1))

In [102]:
# LEAVE AS-IS

print("Some of the movies rated highly by the user who like Harry Potter movies")
display(get_rated_movies(user_movie, movies, user_id_hp).sort_values(by='rating', ascending=False).head(20))

print("Some of the movies rated highly by the user who like dark movies")
display(get_rated_movies(user_movie, movies, user_id_dark).sort_values(by='rating', ascending=False).head(20))


Some of the movies rated highly by the user who like Harry Potter movies


,movie_id,rating,title,genres
0,4963,5.0,Ocean's Eleven (2001),Crime|Thriller
1,5254,5.0,Blade II (2002),Action|Horror|Thriller
2,4995,5.0,"Beautiful Mind, A (2001)",Drama|Romance
3,4993,5.0,"Lord of the Rings: The Fellowship of the Ring,...",Adventure|Fantasy
4,6503,5.0,Charlie's Angels: Full Throttle (2003),Action|Adventure|Comedy|Crime|Thriller
5,6383,5.0,"2 Fast 2 Furious (Fast and the Furious 2, The)...",Action|Crime|Thriller
6,6333,5.0,X2: X-Men United (2003),Action|Adventure|Sci-Fi|Thriller
7,5349,5.0,Spider-Man (2002),Action|Adventure|Sci-Fi|Thriller
8,5418,5.0,"Bourne Identity, The (2002)",Action|Mystery|Thriller
9,5810,5.0,8 Mile (2002),Drama


Some of the movies rated highly by the user who like dark movies


,movie_id,rating,title,genres
0,3535,5.0,American Psycho (2000),Crime|Horror|Mystery|Thriller
1,3949,5.0,Requiem for a Dream (2000),Drama
2,4011,5.0,Snatch (2000),Comedy|Crime|Thriller
3,48394,5.0,"Pan's Labyrinth (Laberinto del fauno, El) (2006)",Drama|Fantasy|Thriller
4,4226,5.0,Memento (2000),Mystery|Thriller
5,4878,5.0,Donnie Darko (2001),Drama|Mystery|Sci-Fi|Thriller
6,4973,5.0,"Amelie (Fabuleux destin d'Amélie Poulain, Le) ...",Comedy|Romance
7,4995,5.0,"Beautiful Mind, A (2001)",Drama|Romance
8,7022,5.0,Battle Royale (Batoru rowaiaru) (2000),Action|Drama|Horror|Thriller
9,7438,5.0,Kill Bill: Vol. 2 (2004),Action|Drama|Thriller


<font size="+1" color="#2680EB">**Describe how you would characterize the tastes of these two users.**</font>

**Answer:**

How would you characterize the tastes of the first user?

* The first user seems to enjoy mainly adventure, fantasy and action movies. This can be seen from the high ratings given to movies such as Harry Potter, The Lord of the Rings and Spider-Man. The user also enjoys science fiction, thrillers and other genres, so their tastes are quite varied. However, as we can see, action and adventure movies predominate.

How would you characterize the tastes of the second user?

* The second user seems to enjoy dark and intense movies, especially dramas and thrillers movies. This can be seen from the high ratings given to movies such as American Psycho, Requiem for a Dream and Donnie Darko. The user also enjoys movies about crime, mystery and science fiction.

## 3.2. Your selection <font color="#2680EB">[10%]</font>

<font size="+1" color="#2680EB">**Implement your code to find these users.** Make sure the titles of the movies that you are using to guide this search are visible in your code. You do not need to do this exactly as we did above, but just make sure you select two users that have clearly different tastes.</font>

In [103]:
# YOUR CODE HERE
#AI
# I define the movie titles I will use to select two users with different tastes.

movies_user_1 = ["Finding Nemo (2003)","Shrek (2001)","Monsters, Inc. (2001)"] # First group: animation and family movies.
movies_user_2 = ["Saw (2004)","Ring, The (2002)", "Grudge, The (2004)"]# Second group: horror movies.
# I get the IDs of the movies
ids_user_1 = [get_movie_id_by_title(title, movies) for title in movies_user_1] 
ids_user_2 = [get_movie_id_by_title(title, movies) for title in movies_user_2]

# I check that all movie titles have been found.
if None in ids_user_1 or None in ids_user_2: raise ValueError("Alguna película no se ha encontrado.") 

min_rating = 4.0 #minimum rating to consider that a user likes a movie


# I find users who rated all three movies in the first group above the minimum rating
users_1 = user_movie[(user_movie[ids_user_1[0]] > min_rating) &(user_movie[ids_user_1[1]] > min_rating) &(user_movie[ids_user_1[2]] > min_rating)].index.tolist()
# I find users who rated all three movies in the second group above the minimum rating
users_2 = user_movie[(user_movie[ids_user_2[0]] > min_rating) &(user_movie[ids_user_2[1]] > min_rating) &(user_movie[ids_user_2[2]] > min_rating)].index.tolist()

# show how many users meet each condition.
print("Users who like the first group:", len(users_1))
print("Users who like the second group:", len(users_2))

if len(users_1) == 0 or len(users_2) == 0: raise ValueError("No se han encontrado usuarios suficientes.")
#I select one user from each group  (I frist do it randomly and the I fixed their IDs) 
user_id_1 = 194321
user_id_2 = 85470

print("Selected first user: user_id =", user_id_1)
print("Selected second user: user_id =", user_id_2)

# I show the 20 highest-rated movies for each user.
print("Movies highly rated by the first user:")
display(get_rated_movies(user_movie, movies, user_id_1).sort_values(by='rating', ascending=False).head(20))

print("Movies highly rated by the second user:")
display(get_rated_movies(user_movie, movies, user_id_2).sort_values(by='rating', ascending=False).head(20))

Users who like the first group: 360
Users who like the second group: 5
Selected first user: user_id = 194321
Selected second user: user_id = 85470
Movies highly rated by the first user:


,movie_id,rating,title,genres
0,4027,5.0,"O Brother, Where Art Thou? (2000)",Adventure|Comedy|Crime
1,4014,5.0,Chocolat (2000),Drama|Romance
2,4993,5.0,"Lord of the Rings: The Fellowship of the Ring,...",Adventure|Fantasy
3,4973,5.0,"Amelie (Fabuleux destin d'Amélie Poulain, Le) ...",Comedy|Romance
4,4306,5.0,Shrek (2001),Adventure|Animation|Children|Comedy|Fantasy|Ro...
5,4886,5.0,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy
6,7153,5.0,"Lord of the Rings: The Return of the King, The...",Action|Adventure|Drama|Fantasy
7,6385,5.0,Whale Rider (2002),Drama
8,6377,5.0,Finding Nemo (2003),Adventure|Animation|Children|Comedy
9,5952,5.0,"Lord of the Rings: The Two Towers, The (2002)",Adventure|Fantasy


Movies highly rated by the second user:


,movie_id,rating,title,genres
0,4148,5.0,Hannibal (2001),Horror|Thriller
1,3897,5.0,Almost Famous (2000),Drama
2,4166,5.0,Series 7: The Contenders (2001),Action|Drama
3,4973,5.0,"Amelie (Fabuleux destin d'Amélie Poulain, Le) ...",Comedy|Romance
4,4878,5.0,Donnie Darko (2001),Drama|Mystery|Sci-Fi|Thriller
5,4642,5.0,Hedwig and the Angry Inch (2000),Comedy|Drama|Musical
6,5617,5.0,Secretary (2002),Comedy|Drama|Romance
7,5666,5.0,"Rules of Attraction, The (2002)",Comedy|Drama|Romance|Thriller
8,5785,5.0,Jackass: The Movie (2002),Action|Comedy|Documentary
9,5418,5.0,"Bourne Identity, The (2002)",Action|Mystery|Thriller


<font size="+1" color="#2680EB">**Describe how you would characterize their tastes** based on the ratings they give to different movies.</font>

**Answer:**

How would you characterize the tastes of your first user, having `user_id=user_id_1`?

* First user (194321): seems to enjoy animation, family and fantasy movies, such as Shrek, Monsters, Inc. and The Lord of the Rings. However, this user also gives high ratings to dramas, comedies and romantic movies, so their tastes are varied.

How would you characterize the tastes of your second user, having `user_id=user_id_2`?

* Second user (85470): seems to enjoy horror, thriller and mystery movies, such as Hannibal, Final Destination 2 and Saw. This user also gives high ratings to dramas, comedies, romantic movies and fantasy movies. Therefore, although this user meets the selection criteria for horror movies, their tastes are not limited to this genre.

# 4. Generate recommendations

## 4.1. Compute item similarities <font color="#2680EB">[5%]</font>

<font size="+1" color="#2680EB">**Compute all Pearson correlations between columns in "user_movie".** Restrict this to columns having at least 100 elements in common.</font>

In [ ]:
# YOUR CODE HERE
#AI
#I calculate the Pearson correlations between all movies.
#min_periods=100 requires at least 100 users to have rated both movies before calculating their correlation.
item_similarity = user_movie.corr(method='pearson',min_periods=100)

# Muestro las primeras cinco filas de la matriz de similitudes.
display(item_similarity.head(5))
item_similarity.head(5)

movie_id,2769,3177,3190,3225,3228,3239,3273,3275,3276,3279,...,51187,51194,51255,51312,51314,51317,51402,51412,51418,51433
movie_id,,,,,,,,,,,,,,,,,,,,,
2769,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3177,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3190,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3225,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3228,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


movie_id,2769,3177,3190,3225,3228,3239,3273,3275,3276,3279,...,51187,51194,51255,51312,51314,51317,51402,51412,51418,51433
movie_id,,,,,,,,,,,,,,,,,,,,,
2769,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3177,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3190,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3225,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3228,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 4.2. Compute the relevance of movies not yet rated by a user <font color="#2680EB">[20%]</font>

<font size="+1" color="#2680EB">**Implement the movies relevance function.** It should return a dataframe with columns "movie_id", "relevance".</font>

In [105]:
def get_movies_relevance(user_id, user_movie, item_similarity):
    
    # Create an empty series
    movies_relevance = pd.Series(dtype='float64')
    
    # Iterate through the movies the user has watched
    rated_movies = get_rated_movie_ids(user_movie, user_id)
    
    for rated_movie in rated_movies:
        # YOUR CODE HERE
        # 1. Obtain the rating given to the rated_movie
        rating = get_rating(user_movie, user_id, rated_movie)
        # 2. Obtain the similarities to all other movies
        similarities = item_similarity[rated_movie]
        # 3. Multiply these into weighted_similarities
        weighted_similarities = similarities * rating
        # Append these terms to movies_relevance
        movies_relevance = pd.concat([movies_relevance, weighted_similarities])
    
    # Compute the sum of weights for each movie,
    # remember that the series is indexed by movie_id
    movies_relevance = movies_relevance.groupby(movies_relevance.index).sum()
    
    # Convert to a dataframe
    # The instructor used AI to convert the series to a dataframe and reindex (Gemini 3.6 Flash)
    movies_relevance_df = pd.DataFrame(movies_relevance, columns=['relevance']).reset_index(names='movie_id')
    
    # Remove movies that the user has rated
    rows_to_remove = movies_relevance_df['movie_id'].isin(rated_movies)
    movies_relevance_df = movies_relevance_df[~rows_to_remove]
    
    return movies_relevance_df

## 4.3. Generate recommendations for your two users <font color="#2680EB">[10%]</font>

In [106]:
# LEAVE AS=IS

print(f"User 1 (user_id={user_id_1})")
rel = get_movies_relevance(user_id_1, user_movie, item_similarity)
rel = rel.merge(movies, on='movie_id')
display(rel.sort_values('relevance', ascending=False).head(10))

print(f"User 2 (user_id={user_id_2})")
rel = get_movies_relevance(user_id_2, user_movie, item_similarity)
rel = rel.merge(movies, on='movie_id')
display(rel.sort_values('relevance', ascending=False).head(10))

User 1 (user_id=194321)


,movie_id,relevance,title,genres
553,5218,36.056777,Ice Age (2002),Adventure|Animation|Children|Comedy
2815,50872,35.799220,Ratatouille (2007),Animation|Children|Drama
917,6218,34.366757,Bend It Like Beckham (2002),Comedy|Drama|Romance
1149,6863,33.524997,School of Rock (2003),Comedy|Musical
2290,40815,33.424605,Harry Potter and the Goblet of Fire (2005),Adventure|Fantasy|Thriller|IMAX
1519,8873,33.034393,"Motorcycle Diaries, The (Diarios de motociclet...",Adventure|Drama
461,4896,33.024501,Harry Potter and the Sorcerer's Stone (a.k.a. ...,Adventure|Children|Fantasy
792,5816,32.978584,Harry Potter and the Chamber of Secrets (2002),Adventure|Fantasy
1565,8970,32.172117,Finding Neverland (2004),Drama
2312,41566,32.166007,"Chronicles of Narnia: The Lion, the Witch and ...",Adventure|Children|Fantasy


User 2 (user_id=85470)


,movie_id,relevance,title,genres
607,5459,183.969961,Men in Black II (a.k.a. MIIB) (a.k.a. MIB 2) (...,Action|Comedy|Sci-Fi
85,3793,182.143129,X-Men (2000),Action|Adventure|Sci-Fi
1192,7254,178.580273,The Butterfly Effect (2004),Drama|Sci-Fi|Thriller
931,6373,177.891526,Bruce Almighty (2003),Comedy|Drama|Fantasy|Romance
1989,34319,171.760382,"Island, The (2005)",Action|Sci-Fi|Thriller
1116,6934,171.690715,"Matrix Revolutions, The (2003)",Action|Adventure|Sci-Fi|Thriller|IMAX
304,4367,170.899812,Lara Croft: Tomb Raider (2001),Action|Adventure
912,6333,167.852955,X2: X-Men United (2003),Action|Adventure|Sci-Fi|Thriller
2276,44022,166.764268,Ice Age 2: The Meltdown (2006),Adventure|Animation|Children|Comedy
518,5218,165.541086,Ice Age (2002),Adventure|Animation|Children|Comedy


<font size="+1" color="#2680EB">**Comment on these recommendations.** Indicate if they are relevant in general, if there is any recommendation that looks particularly correct or particularly wrong for a user, and any other comment you may have.</font>

**Answer:**

What are your comments on the recommendations for your first user, having `user_id=user_id_1`? 

* The recommendations seem quite consistent with this user's tastes. Movies such as Ice Age, Ratatouille, Harry Potter and The Chronicles of Narnia match their interest in animation, fantasy and adventure. There are also comedy and drama movies, such as Bend It Like Beckham and School of Rock, which shows that the recommendations are quite varied.

What are your comments on the recommendations for your first user, having `user_id=user_id_2`?

* The recommendations include movies such as Men in Black II, X-Men, The Butterfly Effect and The Matrix Revolutions. Most of them belong to science fiction, action, adventure and thriller genres. These recommendations seem consistent with the user's interest in dark and intense movies, although they also include some comedy and fantasy elements.

<font size="+2" color="#2680EB">Additional results: surprise library</font> 


In [ ]:
%pip install scikit-surprise

   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   -------------------------------------- - 1.3/1.3 MB 8.5 MB/s eta 0:00:01
   ---------------------------------------- 1.3/1.3 MB 5.8 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [118]:
#AI
from surprise import Dataset, Reader, SVD

# We select the user ID, movie ID and rating columns.
ratings_surprise = ratings[['user_id', 'movie_id', 'rating']]
#We define the rating scale used in the dataset.
reader = Reader(rating_scale=(0.5, 5.0))
# We convert the DataFrame into a Surprise dataset.
data_surprise = Dataset.load_from_df(ratings_surprise,reader)

# We create the SVD model with a fixed random state to get reproducible results.
algo = SVD(random_state=42)

# We use all the available ratings to train the model.
trainset = data_surprise.build_full_trainset()
algo.fit(trainset)

In [ ]:
#AI
# We get the movies that each user has already rated.
rated_movies_1 = set(ratings.loc[ratings['user_id'] == user_id_1, 'movie_id'])
rated_movies_2 = set(ratings.loc[ratings['user_id'] == user_id_2, 'movie_id'])

# We get the movies that each user has not rated yet and we only consider movies that have ratings in the dataset.
unrated_movies_1 = movies[movies['movie_id'].isin(ratings['movie_id'].unique()) &~movies['movie_id'].isin(rated_movies_1)]
unrated_movies_2 = movies[movies['movie_id'].isin(ratings['movie_id'].unique()) &~movies['movie_id'].isin(rated_movies_2)]

# We predict the ratings for the unrated movies.
predictions_1 = [(movie_id, algo.predict(user_id_1, movie_id).est)for movie_id in unrated_movies_1['movie_id']]
predictions_2 = [(movie_id, algo.predict(user_id_2, movie_id).est)for movie_id in unrated_movies_2['movie_id']]

# We create DataFrames with the predicted ratings and movie information.
recommendations_1 = pd.DataFrame(predictions_1,columns=['movie_id', 'estimated_rating']).merge(movies, on='movie_id')
recommendations_2 = pd.DataFrame(predictions_2,columns=['movie_id', 'estimated_rating']).merge(movies, on='movie_id')

# We display the 10 movies with the highest predicted ratings for each user
print(f"Recommendations for user {user_id_1}:")
display(recommendations_1.sort_values('estimated_rating', ascending=False).head(10))

print(f"Recommendations for user {user_id_2}:")
display(recommendations_2.sort_values('estimated_rating', ascending=False).head(10))

Recommendations for user 194321:


,movie_id,estimated_rating,title,genres
1899,31437,4.785637,Nobody Knows (Dare mo shiranai) (2004),Drama
1136,6772,4.742174,To Be and to Have (Être et avoir) (2002),Documentary
303,4308,4.738170,Moulin Rouge (2001),Drama|Musical|Romance
1372,7767,4.726385,"Best of Youth, The (La meglio gioventù) (2003)",Drama
936,6269,4.693114,Stevie (2002),Documentary
1400,8327,4.691438,Dolls (2002),Drama|Romance
1195,6942,4.685038,Love Actually (2003),Comedy|Drama|Romance
1803,27834,4.647725,"Return, The (Vozvrashcheniye) (2003)",Drama
2604,47491,4.637067,Adam's Apples (Adams æbler) (2005),Comedy|Drama
682,5515,4.635532,Songs From the Second Floor (Sånger från andra...,Drama


Recommendations for user 85470:


,movie_id,estimated_rating,title,genres
877,6269,4.870255,Stevie (2002),Documentary
815,6016,4.856472,City of God (Cidade de Deus) (2002),Action|Adventure|Crime|Drama|Thriller
1144,7096,4.855027,Rivers and Tides (2001),Documentary
912,6333,4.848389,X2: X-Men United (2003),Action|Adventure|Sci-Fi|Thriller
2617,50658,4.824048,49 Up (2005),Documentary
316,4380,4.788911,"Princess and the Warrior, The (Krieger und die...",Drama|Romance
2489,48385,4.758285,Borat: Cultural Learnings of America for Make ...,Comedy
1481,9010,4.755934,Love Me If You Dare (Jeux d'enfants) (2003),Drama|Romance
1540,27423,4.754352,"O Auto da Compadecida (Dog's Will, A) (2000)",Adventure|Comedy
1525,27366,4.729623,Werckmeister Harmonies (Werckmeister harmóniák...,Drama



**Observations:** If we compare these results with the results from section 4.3, we can see that they generate different recommendations for the same users. In 4.3 recommends movies that are more closely related to the users' previously rated movies. For example, user1 receives recommendations such as Ice Age, Ratatouille and Harry Potter. With SVD we can see that include more dramas, romantic movies and documentaries. For user2, both methods recommend some action and thriller movies, but SVD also includes other genres, such as documentaries. 

Another interesting point is that SVD gives relatively high estimated ratings to some less popular movies. These predictions are based on the patterns learned from users' ratings, rather than only on the average rating of each movie. However, they are predictions made by the model, not actual ratings from the users. This shows that SVD can identify different preferences and recommend movies that are not necessarily the most popular ones. It also shows that different recommendation algorithms can produce different results, even when they use the same rating data.


<font size="+2" color="#00A365">I hereby declare that I completed this practice myself, that I have not copied it from others, or from external sources without citing them, and that I have followed the AI instructions stated above.</font>